In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_raw=pd.read_csv("flower_3class.csv")

\begin{flalign*}
L_i 
&= -\log\left(p_{i,y_i}\right) \\[6pt]
&= -\sum_{k=1}^{K} y_{ik}\log\left(p_{ik}\right) \\[10pt]
J(W) 
&= \frac{1}{m}\sum_{i=1}^{m} L_i \\[6pt]
&= \frac{1}{m}\sum_{i=1}^{m}
\left[
-\sum_{k=1}^{K} y_{ik}\log\left(p_{ik}\right)
\right] \\[6pt]
&= -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{K}
y_{ik}\log\left(p_{ik}\right)
\end{flalign*}

#梯度推导
x = 一个样本的特征向量\
z = xW = 每个类别的原始得分\
p = softmax(z) = 每个类别的概率\
y = one-hot 真实标签\
\
1.Softmax 是：\
$$p_k = \frac{e^{z_k}}{\sum_{r=1}^{K} e^{z_r}}$$
2.单个样本的损失是：\
$$L = -\sum_{k=1}^{K} y_k \log(p_k)$$
3.链式法则：W → z → p → loss\
\
(1).求$\frac{\partial L}{\partial z_s}$:\
\
展开$log(p_k)$:\
\begin{flalign*}
p_k 
&= \frac{e^{z_k}}{\sum_{r=1}^{K}e^{z_r}} \\[4pt]
\log(p_k)
&= \log\left(\frac{e^{z_k}}{\sum_{r=1}^{K}e^{z_r}}\right) \\[4pt]
&= z_k - \log\left(\sum_{r=1}^{K}e^{z_r}\right)
\end{flalign*}
\
代回损失函数：\
\begin{flalign*}
L
&= -\sum_{k=1}^{K} y_k\log(p_k) \\[4pt]
&= -\sum_{k=1}^{K} y_k
\left[
z_k - \log\left(\sum_{r=1}^{K}e^{z_r}\right)
\right] \\[4pt]
&= -\sum_{k=1}^{K} y_k z_k
+
\sum_{k=1}^{K} y_k
\log\left(\sum_{r=1}^{K}e^{z_r}\right)
\end{flalign*}\
因为 y 是 one-hot，所以：\
$$\sum_{k=1}^{K} y_k = 1$$\
于是：\
\begin{flalign*}
L
&= -\sum_{k=1}^{K} y_k z_k
+
\log\left(\sum_{r=1}^{K}e^{z_r}\right)
\end{flalign*}\
现在对某一个类别得分$z_s$求导：\
\begin{flalign*}
\frac{\partial L}{\partial z_s}
&= \frac{\partial}{\partial z_s}
\left[
-\sum_{k=1}^{K} y_k z_k
+
\log\left(\sum_{r=1}^{K}e^{z_r}\right)
\right] \\[6pt]
&= -y_s
+
\frac{e^{z_s}}{\sum_{r=1}^{K}e^{z_r}} \\[6pt]
&= -y_s + p_s \\[6pt]
&= p_s - y_s
\end{flalign*}
\
(2)从$z$传到$W$:\
$$z_s = x_0w_{0s} + x_1w_{1s} + \cdots + x_nw_{ns}$$\
$$\frac{\partial z_s}{\partial w_{js}} = x_j$$\
链式法则：
\begin{flalign*}
\frac{\partial L}{\partial w_{js}}
&= \frac{\partial L}{\partial z_s}
\cdot
\frac{\partial z_s}{\partial w_{js}} \\[6pt]
&= (p_s - y_s)x_j
\end{flalign*}\
意思是：\
第j个特征对第s个类别的梯度等于第 j 个特征值 × 这个类别的预测误差\
\
3.对m个样本取平均\
\
\begin{flalign*}
\frac{\partial J}{\partial w_{js}}
&= \frac{1}{m}\sum_{i=1}^{m}
x_{ij}(p_{is}-y_{is})
\end{flalign*}\
写成矩阵形式\
\begin{flalign*}
\frac{\partial J}{\partial W}
&= \frac{1}{m}X^T(P-Y)
\end{flalign*}


In [ ]:
'''
X.shape = (m, n)
W.shape = (n, K)
Z.shape = (m, K)
P.shape = (m, K)
Y.shape = (m, K)
'''

In [4]:
df=df_raw.copy()
df.head()

,petal_length,petal_width,species
0,3.79,0.89,0
1,6.17,3.36,1
2,3.04,2.05,1
3,3.00,1.50,0
4,7.14,1.82,1


In [ ]:

cols_x = ['petal_length', 'petal_width']
cols_y = 'species'

X = df[cols_x].values.astype(float)
Y = df[cols_y].values.astype(int)

# =====================
# 2. 划分训练集和测试集
# =====================

np.random.seed(42)

idx = np.random.permutation(len(Y))

train_size = int(0.8 * len(Y))

train_idx = idx[:train_size]
test_idx = idx[train_size:]

X_train = X[train_idx]
X_test = X[test_idx]

Y_train = Y[train_idx]
Y_test = Y[test_idx]

# =====================
# 3. 标准化
# 只用训练集的均值和标准差
# =====================

X_train_raw = X[train_idx]
X_test_raw = X[test_idx]

means = X_train_raw.mean(axis=0)
stds = X_train_raw.std(axis=0, ddof=0)

X_train_scaled = (X_train_raw - means) / stds
X_test_scaled = (X_test_raw - means) / stds

X_train = np.hstack([
    np.ones((X_train_scaled.shape[0], 1)),
    X_train_scaled
])

X_test = np.hstack([
    np.ones((X_test_scaled.shape[0], 1)),
    X_test_scaled
])

# =====================
# 5. one-hot 编码
# =====================

K = len(np.unique(Y))      # 类别数量，这里是 3
n = X_train.shape[1]       # 特征数量 + 截距

Y_train_onehot = np.eye(K)[Y_train]

# 例如：
# Y_train = [0, 2, 1]
# Y_train_onehot =
# [[1, 0, 0],
#  [0, 0, 1],
#  [0, 1, 0]]

# =====================
# 6. softmax 函数
# =====================

def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

# =====================
# 7. 初始化参数
# =====================

W = np.zeros((n, K))

alpha = 0.5
n_iters = 5000

cost_history = []
grad_history = []

# =====================
# 8. 梯度下降
# =====================

for i in range(n_iters):
    # 线性部分
    Z = X_train @ W

    # softmax 概率
    P = softmax(Z)

    # 交叉熵损失
    cost = -np.mean(np.sum(Y_train_onehot * np.log(P + 1e-15), axis=1))
    cost_history.append(cost)

    # 误差
    err = P - Y_train_onehot

    # 梯度
    grad = X_train.T @ err / m
    grad_history.append(np.linalg.norm(grad))

    # 更新参数
    W = W - alpha * grad

# =====================
# 9. 预测函数
# =====================

def predict_proba(X):
    Z = X @ W
    P = softmax(Z)
    return P

def predict(X):
    P = predict_proba(X)
    return np.argmax(P, axis=1)

# =====================
# 10. 评估
# =====================

train_pred = predict(X_train)
test_pred = predict(X_test)

train_acc = np.mean(train_pred == Y_train)
test_acc = np.mean(test_pred == Y_test)

print("训练集准确率:", train_acc)
print("测试集准确率:", test_acc)
print("最终损失:", cost_history[-1])
print("参数矩阵 W:")
print(W)